In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np

In [2]:
corpus = [
    "The cat sat on the mat",
    "The dog ran in the park",
    "The bird sang in the tree"
]

In [4]:
tokens = [sentence.lower().split() for sentence in corpus]
counter = Counter()
for sentence in tokens:
    counter.update(sentence)
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}
for word in counter:
    vocab[word] = len(vocab)

print("Vocabulary")
print(vocab)
sequences = []
for sentence in tokens:
    sequence = [vocab.get(word, vocab["<UNK>"]) for word in sentence]
    sequences.append(sequence)

print("\nSequences")
print(sequences)


vocab_size = len(vocab)

Vocabulary
{'<PAD>': 0, '<UNK>': 1, 'the': 2, 'cat': 3, 'sat': 4, 'on': 5, 'mat': 6, 'dog': 7, 'ran': 8, 'in': 9, 'park': 10, 'bird': 11, 'sang': 12, 'tree': 13}

Sequences
[[2, 3, 4, 5, 2, 6], [2, 7, 8, 9, 2, 10], [2, 11, 12, 9, 2, 13]]


In [5]:
window_size = 2
inputs = []
targets = []
for sequence in sequences:
    for i in range(len(sequence)):
        center_word = sequence[i]
        start = max(0, i - window_size)
        end = min(len(sequence), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                inputs.append(center_word)
                targets.append(sequence[j])

X = np.array(inputs)
y = np.array(targets)

print("\nTraining Pairs")

for i in range(len(X)):
    print(X[i], "->", y[i])


Training Pairs
2 -> 3
2 -> 4
3 -> 2
3 -> 4
3 -> 5
4 -> 2
4 -> 3
4 -> 5
4 -> 2
5 -> 3
5 -> 4
5 -> 2
5 -> 6
2 -> 4
2 -> 5
2 -> 6
6 -> 5
6 -> 2
2 -> 7
2 -> 8
7 -> 2
7 -> 8
7 -> 9
8 -> 2
8 -> 7
8 -> 9
8 -> 2
9 -> 7
9 -> 8
9 -> 2
9 -> 10
2 -> 8
2 -> 9
2 -> 10
10 -> 9
10 -> 2
2 -> 11
2 -> 12
11 -> 2
11 -> 12
11 -> 9
12 -> 2
12 -> 11
12 -> 9
12 -> 2
9 -> 11
9 -> 12
9 -> 2
9 -> 13
2 -> 12
2 -> 9
2 -> 13
13 -> 9
13 -> 2


In [6]:
class SkipGramDataset(Dataset):

    def __init__(self, inputs, targets):
        self.inputs = torch.tensor(inputs, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [8]:
dataset = SkipGramDataset(X, y)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [9]:
class SkipGramModel(nn.Module):

    def __init__(self, vocab_size, embedding_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.linear = nn.Linear(
            embedding_size,
            vocab_size
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output = self.linear(embedded)

        return output


In [40]:
embedding_size = 10

model = SkipGramModel(
    vocab_size,
    embedding_size
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

epochs = 100
for epoch in range(epochs):
    total_loss = 0
    for center, context in dataloader:
        optimizer.zero_grad()
        output = model(center)
        loss = criterion(output, context)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} Loss: {total_loss/len(dataloader):.4f}"
        )

Epoch 0 Loss: 2.7428
Epoch 10 Loss: 1.8008
Epoch 20 Loss: 1.6811
Epoch 30 Loss: 1.6545
Epoch 40 Loss: 1.5997
Epoch 50 Loss: 1.6171
Epoch 60 Loss: 1.6021
Epoch 70 Loss: 1.6144
Epoch 80 Loss: 1.5814
Epoch 90 Loss: 1.5737
